In [4]:
!pip install gdown==4.1.0

  Using cached gdown-4.1.0-py3-none-any.whl
  Using cached soupsieve-2.8-py3-none-any.whl.metadata (4.6 kB)
Using cached soupsieve-2.8-py3-none-any.whl (36 kB)


In [5]:
!pip install gdown
!pip install transformers==4.51.1
!pip install transformers-stream-generator==0.0.5
!pip install trl==0.17.0
!pip install tokenizers==0.21.1
!pip install openai
!pip install jinja2
!pip install jsonlines
!pip install decord
!pip install qwen-vl-utils[decord]==0.0.8

  Using cached transformers-4.51.1-py3-none-any.whl.metadata (38 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached hf_xet-1.2.0-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.9 kB)
Using cached transformers-4.51.1-py3-none-any.whl (10.4 MB)
Using cached huggingface_hub-0.36.0-py3-none-any.whl (566 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.5/803.5 kB 49.2 MB/s eta 0:00:00
Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
Using cached hf_xet-1.2.0-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Using cached transformers_stream_generator-0.0.5-py3-none-any.whl
  Using cached trl-0.17.0-py3-none-any.whl.metadata (12 kB)
  Using cached pyarrow-22.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.2 kB)
  Using cached dill-0.4.0-py3-none-any.whl

In [6]:
!export TRANSFORMERS_CACHE=/dev/shm/huggingface_cache
!export HF_HOME=/dev/shm/huggingface

In [17]:
!gdown --id 1NLyviWLTzx0wTNbuEFto6we_uMSecAMl

Access denied with the following error:

 	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses. 

You may still be able to access the file from the browser:

	 https://drive.google.com/uc?id=1NLyviWLTzx0wTNbuEFto6we_uMSecAMl 



In [7]:
import zipfile
import os

# === 1️⃣ 配置路径 ===
zip_path = "reranker_all_data.zip"                              # 下载后的压缩文件
extract_dir = "/dev/shm"            # 解压目标路径（放在高速内存盘）
os.makedirs(extract_dir, exist_ok=True)

# === 2️⃣ 解压文件 ===
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"✅ 文件已解压至: {extract_dir}")

✅ 文件已解压至: /dev/shm


In [ ]:
!cp /dev/shm/train_all.json /dev/shm/test_all.json ./

In [ ]:
命令行使用#TRANSFORMERS_CACHE=/dev/shm/huggingface_cache HF_HOME=/dev/shm/huggingface python infer_batch.py --config mmkd_white_box.json

In [ ]:
import time

while True:
    print("🟢 keep alive ...")
    time.sleep(600)  # 每 60 秒打印一次


🟢 keep alive ...


In [ ]:
TRANSFORMERS_CACHE=/dev/shm/huggingface_cache HF_HOME=/dev/shm/huggingface python train.py --config mmkd_white_box.json

In [28]:
import zipfile
import os

zip_path = "train_all_distill.zip"   # 压缩包路径
extract_dir = "."                    # “.” 表示当前目录

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"✅ 已将 {zip_path} 解压到当前目录: {os.path.abspath(extract_dir)}")


✅ 已将 train_all_distill.zip 解压到当前目录: /home/jovyan/workspace


In [32]:
import json
import os
from tqdm import tqdm

# === 配置路径 ===
input_json = "test_all_distill.json"              # 原始文件名
output_json = "test_all_distill_with_devshm.json" # 输出文件名
prefix = "/dev/shm/"

# === 读取数据 ===
with open(input_json, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"📂 已加载 {len(data)} 条对话")

# === 逐条修改并显示进度 ===
for conv in tqdm(data, desc="🚀 正在更新 image 路径", unit="conv"):
    for msg in conv:
        if msg.get("role") == "user" and isinstance(msg.get("content"), list):
            for item in msg["content"]:
                if item.get("type") == "image" and isinstance(item.get("image"), str):
                    image_path = item["image"]
                    if not image_path.startswith(prefix):
                        item["image"] = os.path.join(prefix, image_path.lstrip("/"))

# === 保存结果 ===
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print(f"✅ 已完成修改并保存至 {output_json}")


📂 已加载 605676 条对话


🚀 正在更新 image 路径: 100%|██████████| 605676/605676 [00:00<00:00, 647162.37conv/s]


✅ 已完成修改并保存至 train_all_distill_with_devshm.json


In [33]:
import zipfile
import os

# === 配置路径 ===
json_path = "train_all_distill_with_devshm.json"   # 要压缩的文件
zip_path = "train_all_distill_with_devshm.zip"     # 输出压缩包名

# === 压缩 ===
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(json_path, arcname=os.path.basename(json_path))

print(f"✅ 已将 {json_path} 压缩为 {os.path.abspath(zip_path)}")


✅ 已将 train_all_distill_with_devshm.json 压缩为 /home/jovyan/workspace/train_all_distill_with_devshm.zip


In [ ]:
#启动训练TRANSFORMERS_CACHE=/dev/shm/huggingface_cache HF_HOME=/dev/shm/huggingface torchrun --nproc_per_node=4 train.py --config mmkd_white_box.json

In [45]:
!pip install deepspeed


  Using cached deepspeed-0.18.1.tar.gz (1.6 MB)
  Preparing metadata (setup.py) ... done
  Created wheel for deepspeed: filename=deepspeed-0.18.1-py3-none-any.whl size=1764292 sha256=2ecb2a4bbe6be4334e911af3b21c40672c07d338d94c554df200ac175c434ea4
  Stored in directory: /home/jovyan/.cache/pip/wheels/e4/41/59/a9d46caf09e118b9276f33e0f6d502a45b1e455296e98a2a11
Successfully built deepspeed


In [36]:
!python -c "import torch; print(torch.version.cuda)"

12.1


In [42]:
!sudo apt-get update
!sudo apt-get install -y cuda-toolkit-12-4



Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Fetched 128 kB in 1s (159 kB/s)
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  alsa-topology-conf alsa-ucm-conf at-spi2-core ca-certificates-java
  cuda-cccl-12-4 cuda-command-line-tools-12-4 cuda-compiler-12-4 cuda-crt-12-4
  cuda-cudart-12-4 cuda-cudart-dev-12-4 cuda-cuobjdump-12-4 cuda-cupti-12-4
  cuda-cupti-dev-12-4 cuda-cuxxfilt-12-4 cuda-documentation-12-4
  cuda-driver-dev-12-4 cuda-gdb-12-4 cuda-libraries-12-4
  cuda-libraries-dev-12-4 cuda-nsight-12-4 cuda-nsigh

In [43]:
!nvcc -V

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Mar_28_02:18:24_PDT_2024
Cuda compilation tools, release 12.4, V12.4.131
Build cuda_12.4.r12.4/compiler.34097967_0


In [44]:
!pip install deepseed

ERROR: Could not find a version that satisfies the requirement deepseed (from versions: none)
ERROR: No matching distribution found for deepseed


In [ ]:
accelerate launch --config_file configs/muti_gpu.yaml train.py --config mmkd_white_box.json

In [2]:
!zip -r train_reranker.zip . -x "*.zip"

  adding: .ipynb_checkpoints/ (stored 0%)
  adding: .ipynb_checkpoints/train_all_distill-checkpoint.json^C



zip error: Interrupted (aborting)


In [3]:
ls

configs/                train_all_distill.json
infer_batch.py          train_all_distill_with_devshm.json
mmkd_white_box.json     train_all_distill_with_devshm.zip
Qwen/                   train_all_distill.zip
reranker_all_data.zip   train_all.json
reranker_prepare.ipynb  train_all_logits.json
result/                 train.py
test_all.json


In [ ]:
!zip -r train_reranker.zip \
  configs \
  Qwen- \
  infer_batch.py \
  mmkd_white_box.json \
  reranker_prepare.ipynb \
  test_all.json \
  train_all_distill_with_devshm.json \
  train_all_distill.json \
  train_all_logits.json \
  train_all.json \
  train.py
